# Lab 7 — PCA & SVD Dimensionality Reduction

**Student:** Hitesh Gupta | **Roll No.:** IC-2k23-39

Compares Principal Component Analysis (PCA) and Singular Value
Decomposition (SVD) for dimensionality reduction on the Iris dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import os

sns.set_style('whitegrid')
os.makedirs('outputs', exist_ok=True)
DATA_DIR = '../datasets'

## 1. Load & Preprocess Data

In [ ]:
iris = pd.read_csv(f'{DATA_DIR}/iris.csv')
X = iris.iloc[:, :-1].values
y = iris.iloc[:, -1].values
print('Original shape:', X.shape)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Scaled shape:', X_scaled.shape)

## 2. PCA — Explained Variance & Components

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

print('Explained variance ratio:', pca.explained_variance_ratio_)
print('Cumulative variance:', np.cumsum(pca.explained_variance_ratio_))

# Choose n_components for 95% variance
n_95 = np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.95) + 1
print('Components for 95% variance:', n_95)

In [ ]:
# Scree plot
plt.figure(figsize=(8, 4))
plt.bar(range(1, len(pca.explained_variance_ratio_) + 1),
        pca.explained_variance_ratio_, alpha=0.7, label='Individual')
plt.plot(range(1, len(pca.explained_variance_ratio_) + 1),
         np.cumsum(pca.explained_variance_ratio_), 'ro-', label='Cumulative')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('PCA Scree Plot — Iris Dataset')
plt.legend()
plt.savefig('outputs/pca_scree.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Visualise PCA Projection

In [ ]:
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
for species in np.unique(y):
    mask = y == species
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1], label=species, alpha=0.7)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%})')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%})')
plt.title('PCA 2-D Projection — Iris')
plt.legend()
plt.savefig('outputs/pca_2d.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. SVD — Manual Implementation

In [ ]:
# SVD: X = U Σ V^T
U, S, Vt = np.linalg.svd(X_scaled, full_matrices=False)
print('U shape:', U.shape)
print('S shape:', S.shape)
print('Vt shape:', Vt.shape)

# Singular values relate to explained variance
explained = S**2 / np.sum(S**2)
print('SVD explained variance:', explained)
print('Cumulative:', np.cumsum(explained))

# Project onto first 2 components
X_svd_2d = U[:, :2] * S[:2]
print('SVD 2-D projection shape:', X_svd_2d.shape)

In [ ]:
# Compare PCA vs SVD projections
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (X_proj, title) in zip(axes, [(X_2d, 'PCA'), (X_svd_2d, 'SVD')]):
    for species in np.unique(y):
        mask = y == species
        ax.scatter(X_proj[mask, 0], X_proj[mask, 1], label=species, alpha=0.7)
    ax.set_title(f'{title} 2-D Projection')
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')
    ax.legend()
plt.tight_layout()
plt.savefig('outputs/pca_vs_svd.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Downstream Classification After Reduction

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y)

results = []
for n in [2, 3, 4]:
    pca_n = PCA(n_components=n)
    Xtr = pca_n.fit_transform(X_train)
    Xte = pca_n.transform(X_test)
    clf = LogisticRegression(max_iter=200)
    clf.fit(Xtr, y_train)
    acc = accuracy_score(y_test, clf.predict(Xte))
    results.append({'n_components': n, 'accuracy': acc})
    print(f'PCA n={n}: accuracy={acc:.4f}')

# Baseline: no reduction
clf_full = LogisticRegression(max_iter=200)
clf_full.fit(X_train, y_train)
acc_full = accuracy_score(y_test, clf_full.predict(X_test))
print(f'No reduction: accuracy={acc_full:.4f}')

## Summary

- PCA finds orthogonal components maximising variance; SVD decomposes
  the data matrix directly — both yield equivalent projections.
- Iris needs only 2 components for ~98% variance.
- Classification accuracy remains high even after 4→2 reduction.
- PCA vs feature selection: PCA creates new composite features;
  feature selection keeps original features.